# 04. Rust End-to-End Parity Validation

This notebook captures the native-backed parity gate for the current worktree.

It summarizes the full `validate_audit.py` run executed with `runtime.stage2_kernel_backend: native`, then reruns explicit `verify`, selected native-kernel parity tests, and a real stage-2 patch probe.

In [1]:
from __future__ import annotations

import importlib
import importlib.util
import hashlib
import json
import os
import subprocess
import sys
from datetime import datetime, UTC
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
AUDIT_CONFIG = REPO_ROOT / 'notebooks' / '03_pystamps_verification.audit.yaml'
AUDIT_JSON = REPO_ROOT / 'inputs_and_outputs' / 'validation_runs' / 'latest_audit.json'
DATASET_STAGE8 = REPO_ROOT / 'inputs_and_outputs' / 'InSAR_dataset_test_stage8diag'
DATASET_MAIN = REPO_ROOT / 'inputs_and_outputs' / 'InSAR_dataset_test'
PROBE_ROOT = REPO_ROOT / 'tmp' / 'rust_parity_validation'

BASE_ENV = {
    **os.environ,
    'OPENBLAS_NUM_THREADS': '1',
    'OMP_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
    'PYTHONPATH': str(REPO_ROOT),
}


def _ts(path: Path) -> str:
    return datetime.fromtimestamp(path.stat().st_mtime, UTC).isoformat()


def run_command(cmd: list[str], *, check: bool = True, env: dict[str, str] | None = None) -> subprocess.CompletedProcess[str]:
    completed = subprocess.run(
        cmd,
        cwd=REPO_ROOT,
        env=env or BASE_ENV,
        text=True,
        capture_output=True,
        check=False,
    )
    print('$', ' '.join(cmd))
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f'command failed with exit code {completed.returncode}')
    return completed


native_spec = importlib.util.find_spec('pystamps.kernels._stage2_native')
native_mod = importlib.import_module('pystamps.kernels._stage2_native')
native_path = Path(native_spec.origin)
src_path = REPO_ROOT / 'src' / 'lib.rs'

pd.DataFrame(
    [
        {'artifact': 'native_module', 'path': str(native_path), 'modified_utc': _ts(native_path)},
        {'artifact': 'rust_source', 'path': str(src_path), 'modified_utc': _ts(src_path)},
        {'artifact': 'audit_config', 'path': str(AUDIT_CONFIG), 'modified_utc': _ts(AUDIT_CONFIG)},
    ]
)


,artifact,path,modified_utc
0,native_module,/shared/home/rdelprete/PythonProjects/AgenticW...,2026-04-07T09:15:32.323502+00:00
1,rust_source,/shared/home/rdelprete/PythonProjects/AgenticW...,2026-04-02T14:08:21.038898+00:00
2,audit_config,/shared/home/rdelprete/PythonProjects/AgenticW...,2026-03-29T15:36:40.546460+00:00


In [2]:
pd.Series(
    {
        'python_executable': sys.executable,
        'python_version': sys.version.split()[0],
        'native_origin': native_spec.origin,
        'native_exports': ', '.join(sorted(name for name in dir(native_mod) if not name.startswith('_'))),
        'dataset_stage8diag_present': DATASET_STAGE8.exists(),
        'dataset_main_present': DATASET_MAIN.exists(),
        'audit_json_present': AUDIT_JSON.exists(),
    }
)


python_executable             /shared/home/rdelprete/PythonProjects/AgenticW...
python_version                                                           3.14.2
native_origin                 /shared/home/rdelprete/PythonProjects/AgenticW...
native_exports                accumulate_weighted_grid, histogram_with_cente...
dataset_stage8diag_present                                                 True
dataset_main_present                                                       True
audit_json_present                                                         True
dtype: object

## Native-backed audit artifact

The expensive audit run was executed before opening this notebook with the command below. The notebook consumes the generated JSON artifact and then reruns the faster follow-up checks.

```bash
OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 PYTHONPATH=. \
  uv run python scripts/validate_audit.py \
    --datasets inputs_and_outputs/InSAR_dataset_test_stage8diag inputs_and_outputs/InSAR_dataset_test \
    --config notebooks/03_pystamps_verification.audit.yaml \
    --output inputs_and_outputs/validation_runs/latest_audit.json
```

In [3]:
audit_payload = json.loads(AUDIT_JSON.read_text(encoding='utf-8'))
audit_summary = pd.DataFrame(
    [
        {
            'dataset': audit['dataset'],
            'status': audit['status'],
            'ok': audit['ok'],
            'checked': audit['checked'],
            'run_root': audit['run_root'],
            'golden_root': audit['golden_root'],
            'run_source': audit['run_source'],
        }
        for audit in audit_payload['audits']
    ]
)
display(audit_summary)
pd.Series(
    {
        'generated_at_utc': audit_payload['generated_at_utc'],
        'completed': audit_payload['completed'],
        'interrupted': audit_payload['interrupted'],
        'ok': audit_payload['ok'],
        'failed_workflows': ', '.join(audit_payload['failed_workflows']) or '<none>',
    }
)


,dataset,status,ok,checked,run_root,golden_root,run_source
0,InSAR_dataset_test_stage8diag,failed,False,14,/shared/home/rdelprete/PythonProjects/AgenticW...,/shared/home/rdelprete/PythonProjects/AgenticW...,generated_full_loop_run_copy
1,InSAR_dataset_test,failed,False,14,/shared/home/rdelprete/PythonProjects/AgenticW...,/shared/home/rdelprete/PythonProjects/AgenticW...,generated_full_loop_run_copy


generated_at_utc    2026-04-07T09:17:45Z
completed                           True
interrupted                        False
ok                                 False
failed_workflows         full_validation
dtype: object

In [4]:
verify_rows: list[dict[str, object]] = []
for audit in audit_payload['audits']:
    verify_cmd = [
        sys.executable,
        '-m',
        'pystamps.cli',
        '--config',
        str(AUDIT_CONFIG),
        'verify',
        '--run',
        str(audit['run_root']),
        '--golden',
        str(audit['golden_root']),
    ]
    completed = run_command(verify_cmd, check=False)
    verify_payload = json.loads(completed.stdout)
    verify_rows.append(
        {
            'dataset': audit['dataset'],
            'returncode': completed.returncode,
            'ok': verify_payload['ok'],
            'checked': verify_payload['checked'],
            'failed_count': len(verify_payload['failed']),
        }
    )

verify_df = pd.DataFrame(verify_rows)
display(verify_df)


$ /shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/.venv/bin/python3 -m pystamps.cli --config /shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/notebooks/03_pystamps_verification.audit.yaml verify --run /shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/inputs_and_outputs/validation_runs/20260407_091745/InSAR_dataset_test_stage8diag_stage2_8 --golden /shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/inputs_and_outputs/InSAR_dataset_test_stage8diag
{
  "ok": false,
  "checked": 14,
  "failed": [
    {
      "path": "PATCH_1/pm1.mat",
      "message": "Value mismatch for key 'C_ps', max_abs=1.93541"
    },
    {
      "path": "PATCH_1/select1.mat",
      "message": "Value mismatch for key 'C_ps2', max_abs=6.14221"
    },
    {
      "path": "PATCH_1/weed1.mat",
      "message": "Shape mismatch for key 'ix_weed': (79229,) != (79227,)"
    },
    {
      "path": "ps2.mat",
      "message": "Shape mismatch for key 'ij': (75583, 3) != (69009, 3)"
    },
  

$ /shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/.venv/bin/python3 -m pystamps.cli --config /shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/notebooks/03_pystamps_verification.audit.yaml verify --run /shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/inputs_and_outputs/validation_runs/20260407_091745/InSAR_dataset_test_stage2_8 --golden /shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/inputs_and_outputs/InSAR_dataset_test
{
  "ok": false,
  "checked": 14,
  "failed": [
    {
      "path": "PATCH_1/pm1.mat",
      "message": "Value mismatch for key 'Nr', max_abs=0.0492425"
    },
    {
      "path": "PATCH_1/select1.mat",
      "message": "Value mismatch for key 'C_ps2', max_abs=6.14221"
    },
    {
      "path": "PATCH_1/weed1.mat",
      "message": "Shape mismatch for key 'ix_weed': (79229,) != (79227,)"
    },
    {
      "path": "ps2.mat",
      "message": "Value mismatch for key 'bperp', max_abs=6.4859"
    },
    {
      "path": "pm2.mat",


,dataset,returncode,ok,checked,failed_count
0,InSAR_dataset_test_stage8diag,1,False,14,13
1,InSAR_dataset_test,1,False,14,13


In [5]:
pytest_expr = (
    'stage2_native_kernels_match_python_reference '
    'or stage4_stage7_stage8_native_kernels_match_python_reference '
    'or stage2_native_generic_matches_python_single_precision'
)
pytest_cmd = [
    sys.executable,
    '-m',
    'pytest',
    '-q',
    'tests/test_kernels_accelerated.py',
    '-k',
    pytest_expr,
]
pytest_completed = run_command(pytest_cmd, check=False)
pd.Series(
    {
        'returncode': pytest_completed.returncode,
        'selection': pytest_expr,
    }
)


$ /shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/.venv/bin/python3 -m pytest -q tests/test_kernels_accelerated.py -k stage2_native_kernels_match_python_reference or stage4_stage7_stage8_native_kernels_match_python_reference or stage2_native_generic_matches_python_single_precision
..F                                                                      [100%]
=================================== FAILURES ===================================
__________ test_stage2_native_generic_matches_python_single_precision __________

    @pytest.mark.skipif(
        importlib.util.find_spec("pystamps.kernels._stage2_native") is None,
        reason="native stage-2 extension not available",
    )
    def test_stage2_native_generic_matches_python_single_precision() -> None:
        rng = np.random.default_rng(21)
        cpxphase = np.exp(1j * rng.normal(size=(5, 7))).astype(np.complex64)
        bperp = rng.normal(size=(5, 7)).astype(np.float32)
    
        expected = ported._ps_topofit_ba

returncode                                                    1
selection     stage2_native_kernels_match_python_reference o...
dtype: object

In [6]:
PROBE_ROOT.mkdir(parents=True, exist_ok=True)
python_probe = PROBE_ROOT / 'python_probe'
native_probe = PROBE_ROOT / 'native_probe'

probe_base = [
    sys.executable,
    'scripts/stage2_patch1_probe.py',
    '--dataset',
    str(DATASET_STAGE8),
    '--patch',
    'PATCH_1',
]



def md5sum(path: Path) -> str:
    digest = hashlib.md5()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def ensure_probe(run_root: Path, backend: str) -> tuple[subprocess.CompletedProcess[str] | None, dict[str, str]]:
    pm1 = run_root / 'PATCH_1' / 'pm1.mat'
    if not pm1.exists():
        completed = run_command(
            probe_base + ['--run-root', str(run_root), '--kernel-backend', backend],
            check=False,
        )
    else:
        completed = None
    payload = {
        'pm1_exists': str(pm1.exists()),
        'pm1_md5': md5sum(pm1) if pm1.exists() else '<missing>',
        'pm1_mtime_ns': str(pm1.stat().st_mtime_ns) if pm1.exists() else '<missing>',
    }
    return completed, payload


python_completed, python_probe_payload = ensure_probe(python_probe, 'python')
native_completed, native_probe_payload = ensure_probe(native_probe, 'native')
probe_df = pd.DataFrame(
    [
        {
            'backend': 'python',
            'returncode': 0 if python_completed is None else python_completed.returncode,
            'reused_existing_run': python_completed is None,
            **python_probe_payload,
        },
        {
            'backend': 'native',
            'returncode': 0 if native_completed is None else native_completed.returncode,
            'reused_existing_run': native_completed is None,
            **native_probe_payload,
        },
    ]
)
display(probe_df)


,backend,returncode,reused_existing_run,pm1_exists,pm1_md5,pm1_mtime_ns
0,python,0,True,True,496d784b91b4ba639cda0fea3debf5c3,1775556632949014347
1,native,0,True,True,40d5979b149e965d1ba3a97699750177,1775557795680111251


In [7]:
summary = pd.Series(
    {
        'audit_ok': bool(audit_payload['ok']),
        'verify_ok': bool(verify_df['ok'].all()) if not verify_df.empty else False,
        'pytest_ok': pytest_completed.returncode == 0,
        'stage2_probe_ok': (
            (python_completed is None or python_completed.returncode == 0)
            and (native_completed is None or native_completed.returncode == 0)
        ),
        'stage2_probe_md5_match': (
            python_probe_payload.get('pm1_exists') == 'True'
            and native_probe_payload.get('pm1_exists') == 'True'
            and python_probe_payload.get('pm1_md5') == native_probe_payload.get('pm1_md5')
        ),
    }
)
display(summary)


audit_ok                  False
verify_ok                 False
pytest_ok                 False
stage2_probe_ok            True
stage2_probe_md5_match    False
dtype: bool